# Temporal model training and evaluation

This notebook is the interactive entry point for the production training workflow. It delegates model construction, leakage-safe temporal validation, calibration, threshold selection, winner selection, and artifact export to the tested Python modules. Default (`Pago_atiempo=0`) is the event class, and public probabilities remain ordered as `[P(default), P(on-time)]`.

The default notebook profile is a CPU smoke run suitable for automation. Set `CDP_NOTEBOOK_PROFILE=full` before starting the kernel for the complete configured comparison. Run outputs, model binaries, and record-level predictions are written under ignored `runs/`; only aggregate results should be published.

In [ ]:
from pathlib import Path
import copy
import json
import os
import sys
from datetime import datetime, timezone

from IPython.display import Image, display
import numpy as np
import pandas as pd

repository_root = Path.cwd().resolve()
while repository_root != repository_root.parent and not (repository_root / 'dataset.csv').exists():
    repository_root = repository_root.parent
if not (repository_root / 'dataset.csv').exists():
    raise FileNotFoundError('Run this notebook from inside the cdp_2026 checkout')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from etl_scripts.src.ft_engineering import (
    chronological_train_test_split,
    load_config,
    read_raw_data,
)
from etl_scripts.src.model_training_evaluation import (
    build_model,
    load_training_config,
    summarize_classification,
    train_and_evaluate,
)

## Reproducible CPU profile

The notebook intentionally uses CPU even on a CUDA workstation. High-resource CUDA experimentation is isolated from the deployable pipeline; the exported object must load and predict without initializing CUDA. The smoke profile keeps one small candidate and one seed per family and is a workflow check, not a benchmark.

In [ ]:
run_profile = os.environ.get('CDP_NOTEBOOK_PROFILE', 'smoke').strip().lower()
if run_profile not in {'smoke', 'full'}:
    raise ValueError('CDP_NOTEBOOK_PROFILE must be smoke or full')

data_config = load_config()
training_config = copy.deepcopy(load_training_config())
training_config['device'] = 'cpu'
training_config['lightgbm_device'] = 'cpu'
if run_profile == 'smoke':
    training_config['seeds'] = training_config['seeds'][:1]
    training_config['benchmark_repeats'] = 1
    training_config['models'] = {
        family: {name: values[:1] for name, values in space.items()}
        for family, space in training_config['models'].items()
    }
    for space in training_config['models'].values():
        if 'n_estimators' in space:
            space['n_estimators'] = [10]
        if 'epochs' in space:
            space['epochs'] = [2]
    training_config['smoke_run'] = True

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = repository_root / 'runs' / f'model_training_notebook_{run_profile}_{run_stamp}'
print(f'Profile: {run_profile}; device: cpu; output: {output_dir}')

## Load once, then preserve chronology

The raw frame is split chronologically before any preparation is fitted. Timestamp ties stay together. The final holdout is not used for candidate search, seed checks, calibration, threshold selection, or winner selection.

In [ ]:
raw_data = read_raw_data(data_config, repository_root / 'dataset.csv')
chronological_split = chronological_train_test_split(raw_data, data_config)
pd.DataFrame([chronological_split.summary])

## Use the shared model factory

Every family in the orchestrated comparison is created through `build_model`. This unfitted preview makes the notebook contract explicit without introducing a second implementation.

In [ ]:
factory_preview = build_model(
    'logistic_regression',
    config=data_config,
    random_state=training_config['seeds'][0],
    device='cpu',
    training_config=training_config,
)
factory_preview

## Train, select, and export

`train_and_evaluate` searches candidates only on expanding training-period folds, checks family finalists across configured seeds, writes the selection before opening the holdout, and exports a raw-record prediction object.

In [ ]:
training_result = train_and_evaluate(
    raw_data,
    config=data_config,
    output_dir=output_dir,
    training_config=training_config,
    device='cpu',
)
selection = json.loads((output_dir / 'selection.json').read_text(encoding='utf-8'))
selection

## Validation summary

Mean default F1 is the search objective. Models within the configured tolerance are ordered by temporal variation, seed variation, CPU batch latency, artifact size, and name. Heuristic and dummy models are references and cannot win.

In [ ]:
validation_columns = [
    'model', 'mean_f1', 'temporal_f1_std', 'seed_f1_std',
    'mean_precision', 'mean_recall', 'cpu_batch_ms',
    'artifact_bytes', 'training_device',
]
training_result.comparison[validation_columns].sort_values(
    ['mean_f1', 'temporal_f1_std'], ascending=[False, True]
).reset_index(drop=True)

## Verify the shared metric contract on frozen predictions

The saved selected-model predictions are summarized again with `summarize_classification`. This is a reporting consistency check after selection—not a second selection step.

In [ ]:
holdout_predictions = pd.read_csv(output_dir / 'holdout_predictions.csv')
selected_predictions = holdout_predictions.loc[
    holdout_predictions['model'] == selection['model']
].copy()
selected_metrics = summarize_classification(
    selected_predictions['target'].to_numpy(),
    selected_predictions['prediction'].to_numpy(),
    selected_predictions['default_probability'].to_numpy(),
    review_fraction=training_config['review_fraction'],
)
saved_metrics = training_result.holdout.set_index('model').loc[selection['model']]
for metric in ('accuracy', 'default_precision', 'default_recall', 'default_f1', 'average_precision', 'roc_auc'):
    assert np.isclose(selected_metrics[metric], saved_metrics[metric])
pd.DataFrame([selected_metrics]).drop(columns='confusion_matrix')

## Comparative charts

The figures are generated from saved validation and frozen-holdout results by the shared reporting code. CPU inference cost is kept separate from training device.

In [ ]:
for figure_path in sorted((output_dir / 'figures').glob('*.png')):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## Artifact handoff

Use `training_result.best_model` for raw-record `predict` and `predict_proba`, or load `best_model.joblib` in a trusted CPU environment with the package versions recorded in `manifest.json`. The holdout has already been inspected in earlier work and remains a diagnostic benchmark, not fresh external validation.

In [ ]:
artifact_inventory = {
    path.name: path.stat().st_size
    for path in sorted(output_dir.iterdir())
    if path.is_file()
}
pd.Series(artifact_inventory, name='bytes').rename_axis('artifact').to_frame()